# 🎯 [OPS04_M05] 방공 유도무기 교전 우선순위 랭킹 및 복합 신뢰도 지수 산출 모듈 구현
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m05_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M05 |
| **미션 주제** | 방공 유도무기 교전 우선순위 랭킹 및 복합 신뢰도 지수 산출 모듈 구현 |
| **소요 시간** | 약 3.0시간 |
| **핵심 산출물** | `src/verification/ops04_confidence_engine.py` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 방공 유도무기 교전 우선순위 랭킹 및 복합 신뢰도 지수 산출 모듈 구현
**소요시간**: 약 3.0시간 | **이번 산출물**: `src/verification/ops04_confidence_engine.py`

### 📋 미션 개요
**다중 지표 기반의 복합 신뢰도 지수(High/Med/Low)로 저신뢰 오경보를 걸러내고, 가상 대응 자원 잔여 현황을 반영한 '승인 대기' 권고 순위 모듈을 구현합니다.**

- **배경 및 필요성:** 위협 점수만 높게 산출된 불확실한 항적을 그대로 경보하면 지휘관의 판단이 흔들리며, 가용 요격 자원이 제한된 포화 상황에서는 자원 제약을 반영한 대응 순위 권고가 필수적입니다.
- **핵심 과업:** 위협 유사도·센서 합의·신호 안정성·운동학 일관성을 가중 합산해 신뢰도 등급을 산출하고 Low 등급을 보류해 오경보 감축률을 측정하며, 20회 잡음 반복 테스트로 등급 일관성을 검증하고 모든 권고 순위를 '승인 대기' 상태로 생성합니다.
- **학습 목표 (습득 역량):** 불확실성 정량화를 위한 복합 신뢰도 지수 설계 역량, 몬테카를로 잡음 반복을 통한 등급 안정성 검증 능력, 자원 제약 기반 권고 의사결정 모델링 역량을 체득합니다.
- **검증 및 산출물:** 오경보율(FAR) 40% 이상 감축, 신뢰도 산출 일관성 90% 이상을 충족한 신뢰도 엔진(`src/verification/ops04_confidence_engine.py`)을 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 다중 품질 신호를 결합한 복합 신뢰도 지수로 불확실한 경보를 등급화해 오경보를 억제하고, 제한된 가상 대응 자원을 반영한 '승인 대기' 권고 순위 엔진 설계 역량을 체득합니다.

**[핵심 과업]** 위협 유사도·센서 합의·신호 안정성·운동학 일관성을 가중 합산해 High/Med/Low 신뢰도 등급을 산출하고 Low 등급 보류를 통한 오경보율 감축률, 20회 잡음 반복 등급 일관성, 자원 제약 기반 권고 순위를 검증합니다.

- **불확실성 정량화 및 신뢰도 등급화 설계 역량:** 단일 위협 점수에 의존하지 않고 센서 합의와 운동학 일관성 등 4대 품질 지표를 가중 합산해 High/Med/Low 임계 기준을 수립하는 신뢰도 엔지니어링 원리를 익힙니다.
- **잡음 민감도 및 오경보 감축 검증 능력:** Low 등급 경보 보류 전후의 오경보율(FAR) 감축 효과를 계산하고, 측정 잡음을 20회 반복 주입해 등급 유지 비율(일관성 90% 이상)을 검증하는 통계적 평가 기술을 습득합니다.
- **자원 제약 반영 HITL 권고 순위 모델링 역량:** 가상 대응 자원의 잔여 수량과 추상 대응 점수를 결합해 우선순위를 제안하되, 모든 권고를 자동 실행 없이 '승인 대기'로 두는 안전 설계 원칙을 체득합니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
방공 포대별 잔여 요격 미사일 수량 및 교전 가능 확률을 반영하여 최적 대응 순위 및 복합 위협 신뢰도 지수를 산출하는 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **복합 신뢰도 지수와 High/Med/Low 등급으로 오경보를 줄이고, 가상 대응 자원 상황을 반영한 '권고' 대응 순위를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **위협 유사도, 센서 합의, 신호 안정성, 운동학 일관성을 가중 합산해 신뢰도 지수를 만들고 등급 기준을 정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/verification/ops04_confidence_engine.py` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 5 코드를 실행해 오경보율 감축률과 등급 일관성을 확인한다.
#### 이 단계의 역할
미션 5의 `confidence()` 모듈을 실행해 복합 신뢰도 지수 기반 오경보율(FAR) 감축률과 20회 잡음 반복 등급 일관성을 확인합니다.

#### 해야 할 일

- 위협 유사도·센서 합의·신호 안정성·운동학 일관성 4대 요소의 가중 합산 구조와 `GRADE`(High/Med/Low) 기준을 파악한다.
- AI를 Co-pilot으로 활용해 Low 등급 보류 전후 오경보율 감축률과 20회 측정 잡음 반복 시 등급 유지 비율 산출 코드를 검토한다.
- 오경보율 감축(13.9% -> 0.0%, 감축률 100%)과 등급 일관성(99.4%, 목표 90% 이상) 달성 여부를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "위협 유사도, 센서 합의, 신호 안정성, 운동학 일관성을 가중 합산해 High/Med/Low 등급을 산출하고, Low 등급 보류 시 오경보율(FAR) 감축률과 20회 가우시안 잡음 반복 시 등급 일관성(%)을 계산하는 confidence() 검증 코드를 검토해줘."

  - **AI 결과 검증 포인트:** AI가 오경보율 감축률을 계산할 때 분모(Baseline 오경보율) 정의가 올바른지, 그리고 잡음 반복 시 매 회 독립 시드(random seed)가 적용되어 일관성 99.4%가 과대 측정되지 않았는지 확인합니다.

#### 핵심 확인 지표

> 오경보율 13.9% -> 0.0% (감축률 100%, 목표 40% 이상), 등급 일관성 99.4% (목표 90% 이상)

#### 학습자가 제출해야 하는 내용

- 미션 5 실행 결과 및 FAR 감축률·등급 일관성 검증 로그
- AI 프롬프트 질의 및 통계 산출식 검증 기록

#### Step 2. 유형별 등급 교차표에서 기만체·스푸핑이 Low로 내려갔는지 본다.
#### 이 단계의 역할
비행체 유형별 신뢰도 등급 교차표를 통해 기만체(`decoy`)와 스푸핑 항적이 Low 등급으로 분류·보류되는지 확인합니다.

#### 해야 할 일

- 실제 비행체 유형(`uav_type`, `fast_type`, `decoy`, `civil`, `bird`)과 산출된 신뢰도 등급(High/Med/Low)의 교차표를 출력한다.
- AI에게 기만체·스푸핑이 Low 등급으로 내려간 이유(센서 합의 부족 또는 운동학 불일치)와 실제 위협(`uav_type`/`fast_type`)의 High/Med 유지율 분석을 요청한다.
- 오경보 억제와 동시에 실제 긴급 위협이 Low로 잘못 보류되지 않았는지 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "비행체 유형별 신뢰도 등급(High/Med/Low) 교차표를 만들고, 기만체와 스푸핑 항적의 Low 등급 배정 비율과 실제 위협(uav_type, fast_type)의 미탐 보류 발생 여부를 점검하는 분석 코드를 작성해줘."

  - **AI 결과 검증 포인트:** 기만체가 Low로 내려갔는지 확인하는 것뿐만 아니라, 실제 위협 표적이 Low로 잘못 강등되어 경보가 누락되지 않았는지(재현율 보존) 반드시 교차 확인합니다.

#### 핵심 확인 지표

> 기만체·스푸핑 Low 등급 분류·보류 확인, 실제 위협 오보류 0건 검증

#### 학습자가 제출해야 하는 내용

- 유형별 신뢰도 등급 교차표 및 미탐 보류 점검 결과
- AI 분석 결과 대조 검증 기록

#### Step 3. 권고 대응 순위표와 자원 부족으로 상급 보고 권고된 건수를 확인한다.
#### 이 단계의 역할
가상 대응 자원(`RESOURCES`) 잔여 수량과 추상 대응 점수를 반영한 권고 순위표에서 모든 항목이 '승인 대기'로 생성되는지 확인합니다.

#### 해야 할 일

- 권고 대응 순위표의 각 항목 상태가 예외 없이 `'승인 대기'`로 설정되어 있는지 검사한다.
- AI에게 가상 대응 자원 고갈 시 '자원 부족 상급 보고 권고'(13건)가 생성되는 로직과 자동 교전 방지 검증 코드를 요청한다.
- 권고 순위표와 상급 보고 권고 건수를 출력해 HITL 원칙 준수 여부를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "가상 대응 자원(RESOURCES) 잔여 수량을 반영해 권고 대응 순위표를 생성하되, 모든 항목의 상태가 '승인 대기'인지 assert로 검증하고 자원 부족으로 상급 보고가 권고된 건수를 요약 출력하는 코드를 작성해줘."

  - **AI 결과 검증 포인트:** AI가 구현한 자원 배정 로직이 자원을 자동으로 차감·확정(자동 승인)해 버리지 않고, 오직 지휘관 승인 대기용 '권고'와 상급 보고 플래그만 생성하는지 코드 레벨에서 확인합니다.

#### 핵심 확인 지표

> 경보 31건 모두 '승인 대기' (자동 승인 0건), 자원 부족 상급 보고 권고 13건

#### 학습자가 제출해야 하는 내용

- 권고 대응 순위표 및 자원 부족 상급 보고 목록 로그
- AI 코드의 자동 승인 차단 여부 검증 기록

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
`GRADE`의 High 기준을 `0.70`에서 `0.85`로 상향했을 때 등급 분포, 오경보 감축률, 잡음 일관성 변화를 비교 분석합니다.

#### 해야 할 일

- `GRADE`의 High 임계값을 `0.70`에서 `0.85`로 수정해 신뢰도 모듈을 재실행한다.
- AI에게 High 기준 상향 시 High→Med 이동 비율과 20회 잡음 반복 시 경계 부근 항적의 등급 흔들림(일관성 변화) 해석을 요청한다.
- 보수적 등급 기준의 장단점(경보 신중도 증가 vs 지휘관 즉시 주목 표적 축소)을 메모로 정리한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "GRADE의 High 임계값을 0.70에서 0.85로 높였을 때 High/Med/Low 등급 분포와 20회 잡음 반복 일관성이 어떻게 변하는지 비교표로 출력하고 전술적 의미를 설명해줘."

  - **AI 결과 검증 포인트:** High 임계값을 지나치게 높여 실제 긴급 표적이 High에서 Med로 내려가면서 지휘관 우선 확인 지연이 생기지 않는지 전술적 영향을 점검합니다.

#### 핵심 확인 지표

> GRADE High 0.70 vs 0.85 등급 분포 및 일관성 비교 분석 완료

#### 학습자가 제출해야 하는 내용

- GRADE 임계값 변경 전후 비교 분석 메모 및 최적 기준 근거
- AI 질의 프롬프트 및 경계 민감도 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 신뢰도 등급(High/Med/Low) 부여 기준의 군사적 타당성

#### 미션 PASS 조건
- 정량 기준: 오경보율(FAR) 40% 이상 감축, 신뢰도 스코어 산출 일관성 90% 이상
- 진행 기준: 오경보 필터링 전후 비교 지표 산출 및 통합 테스트 완료
- 검증 방법: ROC-AUC 곡선 분석 및 임계값 민감도 테스트
- AI 검증 기준: 복합 신뢰도 등급 기준과 '승인 대기' 강제(자동 승인 0건)·20회 잡음 일관성(90% 이상)을 AI 결과 검증을 통해 입증 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
"""위협 유사도·센서 합의·신호 안정성·운동학 일관성을 결합한 복합 신뢰도 지수(High/Med/Low)를 산출해 오경보를 줄이고, 가상 대응 자원의 잔여 수량과 가정 대응 성공 점수를 반영한 '권고' 대응 순위를 만든다(모든 권고는 지휘관 승인 대기)."""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

# ---------------- 위협 점수 모델(추상 모델: 거리·도달시간·자산 가치·위협 유사도) ----------------
FEATS = ["log_spd", "alt_km", "rcs", "rcs_var", "n_sensors", "civ_iff", "jump_ratio"]


def features(df):
    return pd.DataFrame({"log_spd": np.log(df["spd"]), "alt_km": df["alt"] / 1000, "rcs": df["rcs"], "rcs_var": df["rcs_var"],
                         "n_sensors": df["n_sensors"], "civ_iff": df["civ_iff"], "jump_ratio": df["max_jump"] / df["spd"]})


def train_classifier(seeds=(101, 102, 103, 104)):
    """학습용 가상 상황(시험 상황과 다른 시드)으로 '위협 유사도' 분류기를 학습한다."""
    tr = pd.concat([make_raid(s, 120, spoof=10) for s in seeds], ignore_index=True)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    clf.fit(features(tr)[FEATS], tr["threat"].astype(int))
    return clf


def baseline_rank(df):
    """Baseline: 거리·도달시간만으로 1차 위협 점수를 매긴다(자산 가치·유형 판별 없음)."""
    _, tti = geometry(df)
    t_min = tti.min(axis=1)
    out = df[["id"]].copy()
    out["tti_s"] = t_min
    out["score"] = np.exp(-t_min / 300)
    out["urgent_pred"] = t_min <= TTI_URGENT
    return out.sort_values("score", ascending=False)


def advanced_rank(df, clf, w=None):
    """고도화: 자산 가치 x 도달시간 x 위협 유사도, 운동학 불일치(스푸핑 의심) 항적은 별도 표시한다."""
    w = w or {"tau": 300, "p_min": 0.5, "value_pow": 1.0, "jump_max": 2.0}
    _, tti = geometry(df)
    val = ASSETS["value"].to_numpy()[None] ** w["value_pow"]
    asset_score = val * np.exp(-tti / w["tau"])
    best = asset_score.argmax(axis=1)
    p = clf.predict_proba(features(df)[FEATS])[:, 1]
    spoof = (df["max_jump"] / df["spd"]).to_numpy() > w["jump_max"]
    out = df[["id"]].copy()
    out["asset"] = ASSETS["asset"].to_numpy()[best]
    out["tti_s"] = tti[np.arange(len(df)), best]
    out["p_threat"] = p
    out["spoof_flag"] = spoof
    out["score"] = np.where(spoof, 0, p * asset_score.max(axis=1))
    out["urgent_pred"] = (p >= w["p_min"]) & (out["tti_s"] <= TTI_URGENT) & ~spoof
    return out.sort_values("score", ascending=False)

GRADE = [(0.70, "High"), (0.45, "Med"), (0.0, "Low")]
# [교육용가상자료] 가상 대응 자원: 잔여 수량, 담당 자산, 유형별 '가정 대응 성공 점수'(추상값, 실제 무기 성능 아님)
RESOURCES = pd.DataFrame([["R1", 6, ["A1", "A2"], {"uav_type": 0.8, "fast_type": 0.5}],
                          ["R2", 4, ["A1", "A3"], {"uav_type": 0.7, "fast_type": 0.7}],
                          ["R3", 8, ["A2", "A3", "A4"], {"uav_type": 0.6, "fast_type": 0.3}]],
                         columns=["res", "remain", "covers", "score_by_type"])


def confidence(df, clf):
    p = clf.predict_proba(features(df)[FEATS])[:, 1]
    agree = df["n_sensors"].to_numpy() / 3
    stable = 1 - np.clip(df["rcs_var"].to_numpy() / 0.4, 0, 1)
    kin = np.clip(2 - df["max_jump"].to_numpy() / df["spd"].to_numpy(), 0, 1)
    conf = 0.5 * p + 0.2 * agree + 0.15 * stable + 0.15 * kin
    grade = np.select([conf >= GRADE[0][0], conf >= GRADE[1][0]], ["High", "Med"], "Low")
    return conf, grade


clf = train_classifier()
raid = make_raid(8, 80, spoof=6)
base = raid.merge(baseline_rank(raid)[["id", "urgent_pred"]], on="id")
adv = raid.merge(advanced_rank(raid, clf), on="id")
conf, grade = confidence(raid, clf)
adv["conf"] = conf[raid.set_index("id").index.get_indexer(adv["id"])]
adv["grade"] = grade[raid.set_index("id").index.get_indexer(adv["id"])]
alert = adv["urgent_pred"] & (adv["grade"] != "Low")

far_b = (base["urgent_pred"] & ~base["urgent"]).sum() / base["urgent_pred"].sum()
far_n = (alert & ~adv["urgent"]).sum() / max(alert.sum(), 1)
cut = (far_b - far_n) / far_b if far_b > 0 else 1.0
print(f"[오경보율] Baseline {far_b:.1%} -> 신뢰도 엔진 {far_n:.1%}, 감축률 {cut:.0%} (목표 40% 이상) -> {'달성' if cut >= 0.4 else '미달'}")
print("[등급 기준] High: 지수 0.70 이상(복수 센서·안정 신호), Med: 0.45~0.70(추가 확인 필요), Low: 0.45 미만(경보 보류)")
print(pd.crosstab(adv["type"], adv["grade"]).to_string())

# 일관성: 측정 잡음(반사강도·속도 ±5%)을 넣어 20회 반복할 때 등급이 유지되는 비율
rng = np.random.default_rng(0)
same = []
for _ in range(20):
    noisy = raid.copy()
    noisy["rcs"] *= rng.normal(1, 0.05, len(raid)); noisy["spd"] *= rng.normal(1, 0.05, len(raid))
    same.append((confidence(noisy, clf)[1] == grade).mean())
cons = float(np.mean(same))
print(f"\n신뢰도 등급 일관성 {cons:.1%} (목표 90% 이상) -> {'달성' if cons >= 0.9 else '미달'}")

# 권고 대응 순위: 점수 순으로 담당 자원 중 (가정 대응 점수 x 잔여 비율)이 가장 큰 자원을 '권고'
remain = dict(zip(RESOURCES["res"], RESOURCES["remain"]))
recs = []
for r in adv[alert].sort_values("score", ascending=False).itertuples():
    cand = [(row.score_by_type.get(r.type, 0.4) * remain[row.res] / row.remain, row.res) for row in RESOURCES.itertuples()
            if r.asset in row.covers and remain[row.res] > 0]
    res = max(cand)[1] if cand else "자원부족-상급보고"
    if cand:
        remain[res] -= 1
    recs.append({"권고순위": len(recs) + 1, "표적": r.id, "지향자산": r.asset, "도달예상(s)": int(r.tti_s), "신뢰도": round(r.conf, 2),
                 "등급": r.grade, "권고자원": res, "상태": "승인 대기"})
rec = pd.DataFrame(recs)
print(f"\n[권고 대응 순위] 경보 {len(rec)}건 중 상위 10건 (자동 실행 없음, 지휘관 승인 필요)")
print(rec.head(10).to_string(index=False))
print("잔여 수량(가상):", remain, f"| 자원 부족으로 상급 보고 권고 {int((rec['권고자원'] == '자원부족-상급보고').sum())}건")


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[오경보율] Baseline 13.9% -> 신뢰도 엔진 0.0%, 감축률 100% (목표 40% 이상) -> 달성
[등급 기준] High: 지수 0.70 이상(복수 센서·안정 신호), Med: 0.45~0.70(추가 확인 필요), Low: 0.45 미만(경보 보류)
grade      High  Low  Med
type                     
bird          0    8    1
civil         0    0   10
decoy         1    7    2
fast_type    15    0    0
spoof         0    6    0
uav_type     35    0    1

신뢰도 등급 일관성 99.4% (목표 90% 이상) -> 달성

[권고 대응 순위] 경보 31건 중 상위 10건 (자동 실행 없음, 지휘관 승인 필요)
 권고순위     표적 지향자산  도달예상(s)  신뢰도   등급 권고자원    상태
    1 T08023   A1      163 0.96 High   R2 승인 대기
    2 T08044   A2       99 0.90 High   R1 승인 대기
    3 T08034   A1      170 0.94 High   R2 승인 대기
    4 T08015   A3       69 0.94 High   R2 승인 대기
    5 T08036   A1      192 0.96 High   R1 승인 대기
    6 T08027   A2      107 0.89 High   R3 승인 대기
    7 T08058   A2      143 0.97 High   R1 승인 대기
    8 T08006   A2      145 0.87 High   R3 승인 대기
    9 T08068   A3      118 0.98 High   R3 승인 대기
   10 T08003   A2      153 0.90 High   R1 승인 대기
잔여 수량(가상): {'R1': 0, 'R2': 0, 'R3': 0} | 자원 부족으로 상급 보고 권고 13건
```
</details>
